# Educational Plattform Panda Robot
Author: uninh, uqwfa, ultck

## Controller_Task
### Switch_Controller

In this task, you will learn...

> - Understand how to stop a controller and load a new controller.
> - Learn how to activate a new controller.
> - Find out how to switch back to the previous controllers after completing the task

In [ ]:
##### DO NOT CHANGE #####

import rospy
from controller_manager_msgs.srv import LoadController, SwitchController, SwitchControllerRequest, ListControllers
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from moveit_commander import roscpp_shutdown

##### DO NOT CHANGE #####

#### Creating a List of All Active Controllers

In the next section, a list of all active controllers will be created.

The method `wait_for_service` blocks the code until the service `/controller_manager/list_controllers` is available. This ensures that the service is ready before attempting to use it.

After that, a service proxy is created. A service proxy is an object that allows making service calls as if it were a local function. In this case, the service `/controller_manager/list_controllers` is used.

Finally, a list of names of the active controllers is created.

In [ ]:
##### DO NOT CHANGE #####

# create a list of active controllers
rospy.wait_for_service('/controller_manager/list_controllers')
list_controllers = rospy.ServiceProxy('/controller_manager/list_controllers', ListControllers)
active_controllers = [controller.name for controller in list_controllers().controller if controller.state == 'running']

##### DO NOT CHANGE #####

#### Stopping Active Controllers

If there are active controllers, they should be stopped in the following section.

As before, we wait until the service is available. Then, in the try block, a proxy for the ROS service `switch_controller` is created and a new request (`SwitchControllerRequest`) is made. The list of controllers to be stopped in the request is set to the list of all active controllers. Setting the strictness to `BEST_EFFORT` means that the service will do its best to stop the controllers, but it is not guaranteed that all will be stopped. Finally, the response is saved and checked to see if all services were successfully stopped.

In [ ]:
##### DO NOT CHANGE #####

if active_controllers:
    rospy.wait_for_service('/controller_manager/switch_controller')
    try:
        switch_controller = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
        req = SwitchControllerRequest()
        req.stop_controllers = active_controllers
        req.strictness = SwitchControllerRequest.BEST_EFFORT
        response = switch_controller(req)
        assert response.ok, "Active controllers could not be stopped."
    except rospy.ServiceException as e:
        rospy.logerr("Service call failed: %s" % e)

##### DO NOT CHANGE #####

#### Loading a New Controller

Now we load the new controller.

First, the name of the controller to be activated is defined. Then it is checked whether the controller is already loaded. If not, the service `/controller_manager/load_controller` is used to load the controller. If the controller is successfully loaded, this is confirmed with a confirmation message.

In [ ]:
##### DO NOT CHANGE #####

# Define the controller to be activated
controller_to_activate = 'position_joint_trajectory_controller'

# Check if the controller is already loaded
rospy.wait_for_service('/controller_manager/list_controllers')
list_controllers = rospy.ServiceProxy('/controller_manager/list_controllers', ListControllers)
loaded_controllers = [controller.name for controller in list_controllers().controller]
if controller_to_activate not in loaded_controllers:
    # Service for loading the controller to be activated
    rospy.wait_for_service('/controller_manager/load_controller')
    load_service = rospy.ServiceProxy('/controller_manager/load_controller', LoadController)
    response = load_service(controller_to_activate)
    assert response.ok, "Controller could not be loaded."

##### DO NOT CHANGE #####

#### Activating the New Controller

After the new controller has been loaded, it will now be activated.

First, wait until the service `/controller_manager/switch_controller` is available. Then, a service proxy for the service is created. A new request (`SwitchControllerRequest`) is made and the name of the controller to be started is added to the request. The strictness is set to `BEST_EFFORT`, which means that the service will do its best to start the controller. Finally, the response is saved and checked to see if the controller was successfully started.

In [ ]:
##### DO NOT CHANGE #####

# Service for switching to the controller to be activated
rospy.wait_for_service('/controller_manager/switch_controller')
switch_service = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
switch_req = SwitchControllerRequest()
switch_req.start_controllers = [controller_to_activate]
switch_req.strictness = SwitchControllerRequest.BEST_EFFORT
response = switch_service(switch_req)
assert response.ok, "Controller could not be started."

##### DO NOT CHANGE #####

#### Task

Now create a list of all currently active controllers as done in a previous code cell.

**Important**
Assign different identifiers to the individual elements to avoid confusion !!!

- Use the method `wait_for_service` to wait for the service `/controller_manager/list_controllers`
- Then create a service proxy for this service and retrieve the list of active controllers
- Filter the controllers that have the state 'running' and save their names in a list
- Finally, print the list to the terminal

In [ ]:
##### YOUR CODE HERE #####

rospy.wait_for_service('/controller_manager/list_controllers')
list_controllers_1 = rospy.ServiceProxy('/controller_manager/list_controllers', ListControllers)
active_controllers_1 = [controller.name for controller in list_controllers_1().controller if controller.state == 'running']
print(f"Active controllers: {active_controllers_1}")

##### YOUR CODE HERE #####

Finally, we switch back to the previous controllers, which you should now implement yourself.

- Use the method `wait_for_service` to wait for the service `/controller_manager/switch_controller`.
- Then create a service proxy for this service.
- Create a new request (`SwitchControllerRequest`) and add the name of the controller to be stopped (`'position_joint_trajectory_controller'`) to the `stop_controllers` list.
- Add the list of previous active controllers (`active_controllers`) to the `start_controllers` list.
- Set the `strictness` to `BEST_EFFORT`.
- Check if the service was successful.

In [ ]:
##### YOUR CODE HERE #####

# Switch back to previous controllers
rospy.wait_for_service('/controller_manager/switch_controller')
try:
    switch_controller = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
    req = SwitchControllerRequest()
    req.stop_controllers = ['position_joint_trajectory_controller']
    # the controllers that were active before
    req.start_controllers = active_controllers
    req.strictness = SwitchControllerRequest.BEST_EFFORT
    response = switch_controller(req)
    assert response.ok, "Controller konnte nicht gewechselt werden."
except rospy.ServiceException as e:
    rospy.logerr("Service call failed: %s" % e)

##### YOUR CODE HERE #####

### Cleaning up and quitting the ROS node
At the end, the ROS node is shut down and the resources are released.

In [ ]:
##### DO NOT CHANGE #####

# Shutdown the ROS node
rospy.signal_shutdown("Task completed")
roscpp_shutdown()

##### DO NOT CHANGE #####